## Descion Tree From Scratch (Binary Result)

In [66]:
# Import Packages
import pandas as pd
import math
from pprint import pprint # clear dictionay printing

In [67]:
# DataSet Example
dataset = pd.read_csv("./play_tennis_dataSet.csv")
dataset = dataset.drop("day", axis=1)

# Based on Dataset Make Data Structure
newDataSet = {}
for col in dataset:
    colValue = dataset[col]
    newDataSet[col] = colValue.tolist()
print(newDataSet)

{'outlook': ['sunny', 'sunny', 'overcast', 'rain', 'rain', 'overcast', 'sunny', 'sunny', 'rain', 'rain', 'sunny', 'overcast', 'overcast', 'rain'], 'temp': ['hot', 'hot', 'hot', 'mild', 'cool', 'cool', 'mild', 'cool', 'mild', 'cool', 'mild', 'mild', 'cool', 'mild'], 'play': [0, 0, 1, 1, 0, 1, 0, 1, 1, 1, 1, 1, 1, 0]}


## Get Root Element By Calculate Entropy For Eash Feature

In [68]:
goalFeature = "play"

def getFeatureValue(feature, dataset):
    featureValue = dataset[feature]
    newFeatureValue = []
    for i in range(0, len(featureValue)):
        if featureValue[i] not in newFeatureValue: newFeatureValue.append(featureValue[i])
    return newFeatureValue
# getFeatureValue("play")


# Get S for Main DataSet
def getSOfDataSet(dataset):
    S = [dataset[goalFeature].count(1), dataset[goalFeature].count(0)]
    return S


# Get S For Specific Feature
def getSForSpecificFeatureValue(feature, dataset, featureValue):
    colOfFeauture = dataset[feature]
    colOfGoal = dataset[goalFeature]
    # print(colOfFeauture)
    # print(colOfGoal)

    S = []
    countOfNeg = 0
    countOfPos = 0
    dataSetLen = len(dataset[goalFeature])
    for i in range(0, dataSetLen):
        if (colOfFeauture[i] == featureValue):
            if (colOfGoal[i] == 0): countOfNeg += 1
            if (colOfGoal[i] == 1): countOfPos += 1
    
    S = [countOfPos, countOfNeg]
    # print(S)
    return S

# getSForSpecificFeatureValue("outlook", newDataSet, "sunny")

def getEntropyOfS(S):
    sumOfS = sum(S)
    if S[0] == S[1]:
        return 1
    if sumOfS == 0 or S[0] == 0 or S[1] == 0:
        return 0
    p0 = S[0] / sumOfS
    p1 = S[1] / sumOfS
    e = - p0 * math.log2(p0) - p1 * math.log2(p1)
    return round(e, 2)


# Calculate Entropy
def calculateGain(dataset):
    gainOfFeature = []
    for feature in dataset:
        if (feature != goalFeature):
            # print("----- Feature:", feature)
            featureValues = getFeatureValue(feature=feature, dataset=dataset)
            # print(f"featureValues = {featureValues}")
            entropyOfFeatures = []
            for featureValue in featureValues:
                S = getSForSpecificFeatureValue(feature=feature, dataset=dataset, featureValue=featureValue)

                # Caculate Entropy
                e = getEntropyOfS(S)
                entropyOfFeatures.append({featureValue: e, "S": S})
                # print(f"feature: {featureValue} and Entropy = {round(e, 2)} and S = {S}")

            # Calculate Gain Of Feature
            # print(entropyOfFeatures)
            S_total = getSOfDataSet(dataset)
            gain = getEntropyOfS(S_total)
            for entropyFeature in entropyOfFeatures:
                featureValue = list(entropyFeature.keys())[0]
                S_v = entropyFeature['S']
                weight = sum(S_v) / sum(S_total)
                gain -= entropyFeature[featureValue] * weight

            gainOfFeature.append({feature: gain})
    return gainOfFeature

gainOfFeatures = calculateGain(newDataSet)
print(gainOfFeatures)


# Get Root Element
def getRootElement(gainOfFeatures):
    max_val = list(gainOfFeatures[0].values())[0]
    bestFeature = list(gainOfFeatures[0].keys())[0]
    for feature in gainOfFeatures:
        val = list(feature.values())[0]
        if (max_val <= val):
            max_val = val
            bestFeature = list(feature.keys())[0]
    return bestFeature

bestFeature = getRootElement(gainOfFeatures=gainOfFeatures)

print(f"best feature is {bestFeature}")

[{'outlook': 0.2471428571428571}, {'temp': 0.09142857142857136}]
best feature is outlook


## Build a Descion Tree

In [ ]:
# Create Node
def create_node(featureName, featureValue, entropy, S):
    return {
        'featureName': featureName,
        'featureValue': featureValue,
        'entropy': entropy,
        'S': S,
        'children': []
    }


# Add Child to Created Node
def add_child(node, obj):
    node['children'].append(obj)


# Intialize The Tree And return first Node in Tree
def initTree():
    # Get Best Feature
    gainOfFeatures = calculateGain(dataset=newDataSet)
    bestFeature = getRootElement(gainOfFeatures=gainOfFeatures)

    # Calculate S and entropy total
    S_total = getSOfDataSet(dataset=newDataSet)
    entropy_total = getEntropyOfS(S=S_total)

    # Create Root 
    root = create_node(featureName=bestFeature, featureValue="ROOT", entropy=entropy_total, S=S_total)

    # Get Feature Value
    featureValues = getFeatureValue(feature=bestFeature, dataset=newDataSet)

    # Get Entropy For Each Feature
    for value in featureValues:
        S_v = getSForSpecificFeatureValue(feature=bestFeature, dataset=newDataSet, featureValue=value)
        entropy_Sv = getEntropyOfS(S=S_v)

        child = create_node(featureName=bestFeature, featureValue=value, entropy=entropy_Sv, S=S_v)
        add_child(root, child)

    return root

root = initTree()
# pprint(root)


# Get New DataSet From 
def getNewDatasetBasedOnFeatureValue(dataset, featureName, featureVal):
    # print(featureName, featureVal)

    # get All Features in DataSet
    features = list(dataset.keys())
    features.remove(featureName)
    # print(f"Remove Some Elements: FEATURES {features} AND FEATURE ELEMENT {featureName}")
    newDataSet = {}
    # print(features)
    for i in range(0, len(dataset[featureName])):
        if (featureVal == dataset[featureName][i]):
            # print(dataset[featureName][i])
            for f in features:
                # print("######### ", f)
                if f in newDataSet:
                    newDataSet[f].append(dataset[f][i])
                    # print("Should Be Extracted", dataset[f][i])
                else:
                    newDataSet[f] = []
                    newDataSet[f].append(dataset[f][i])

    # print(newDataSet)
    # print("-" * 20)
    return newDataSet


# Build a Tree
def buildTree(dataset, root):
    for child in root["children"]:
        # Extract New Dataset Based on Feature Value
        extractedDataSet = getNewDatasetBasedOnFeatureValue(
            dataset=dataset, 
            featureName=root["featureName"], 
            featureVal=child['featureValue']
        )
        # print("Child: ", child)
        # print("extractedDataSet")
        # pprint(extractedDataSet)

        # If entropy is 0, this is a leaf node - label and stop
        if child["entropy"] == 0:
            child['label'] = 'Yes' if child['S'][0] > 0 else 'No'
            # pprint(child)
            # print("=" * 30)
            continue

        # Check if there are any features left to split on
        remainingFeatures = [f for f in extractedDataSet.keys() if f != goalFeature]
        if not remainingFeatures:
            # No features left, assign majority label
            child['label'] = 'Yes' if child['S'][0] >= child['S'][1] else 'No'
            # print("No features left, assigning majority label:", child['label'])
            continue

        # Get Entropy, S
        S_v = getSOfDataSet(dataset=extractedDataSet)
        entropy_v = getEntropyOfS(S_v)
        # print("S = ", S_v, "Entropy = ", entropy_v)

        # Calculate gain for features using extractedDataSet (not global)
        gainOfFeatures = calculateGain(dataset=extractedDataSet)
        # print("gainOfFeatures = ", gainOfFeatures)
        bestFeature = getRootElement(gainOfFeatures=gainOfFeatures)
        # print("bestFeature is: ", bestFeature)

        # Create Root 
        new_root = create_node(featureName=bestFeature, featureValue="ROOT", entropy=entropy_v, S=S_v)
        add_child(node=child, obj=new_root)

        # Get Feature Value
        featureValues = getFeatureValue(feature=bestFeature, dataset=extractedDataSet)
        
        # Handle The Case of Entropy is 1 and Not Feature Found
        if len(featureValues) == 0:
            if entropy_Sv == 1:
                child["label"] = 'YES'
                break

        for value in featureValues:
            S_v = getSForSpecificFeatureValue(
                feature=bestFeature, 
                dataset=extractedDataSet, 
                featureValue=value
            )
            entropy_Sv = getEntropyOfS(S=S_v)

            # Create new child node
            newChild = create_node(
                featureName=bestFeature, 
                featureValue=value, 
                entropy=entropy_Sv, 
                S=S_v
            )

            # Add child to parent first
            add_child(new_root, newChild)
            
            # print("NEW CHILD FOUNDED")
            if entropy_Sv == 0:
                # Leaf node: assign label
                newChild['label'] = 'Yes' if newChild['S'][0] > 0 else 'No'
                # pprint(newChild)
                # print("=" * 30)
                continue
            
            # pprint(newChild)
            # print("=" * 30)
            

        # Recurse on non-leaf nodes
        # print("<<<<<<<<<<<<< START RECURSION <<<<<<<<<<<<<")
        # print("ENTRY ELEMENT: ")
        # print(f"dataset={extractedDataSet}, root={newChild}")
        buildTree(dataset=extractedDataSet, root=new_root)

        # print("BIG CHILD: ")
        # pprint(child)
        # print("-" * 30)


buildTree(dataset=newDataSet, root=root)
# print("--" * 40)
# pprint(root)


# Display a Tree
def display_tree(node, prefix="", is_last=True, is_root=True):
    # define Some Colors
    RESET   = "\033[0m"
    BOLD    = "\033[1m"
    BLUE    = "\033[94m"
    GREEN   = "\033[92m"
    YELLOW  = "\033[93m"
    RED     = "\033[91m"
    CYAN    = "\033[96m"
    MAGENTA = "\033[95m"

    connector = "┌── " if is_root else ("└── " if is_last else "├── ")
    extension = "    "  if is_last else "│   "

    feature   = node['featureName']
    value     = node['featureValue']
    entropy   = node['entropy']
    S         = node['S']
    children  = node['children']
    label     = node.get('label', None)

    # Format each part
    if is_root:
        feature_str = f"{BOLD}{BLUE}[{feature}]{RESET}"
    else:
        feature_str = f"{CYAN}{feature}{RESET}"

    if value == "ROOT":
        value_str = f"{BOLD}{MAGENTA}ROOT{RESET}"
    else:
        value_str = f"{YELLOW}'{value}'{RESET}"

    entropy_color = RED if entropy == 1 else (GREEN if entropy == 0 else YELLOW)
    entropy_str   = f"{entropy_color}H={entropy}{RESET}"

    S_str = f"{BLUE}S=[{GREEN}{S[0]}Yes{RESET},{BLUE} {RED}{S[1]}No{RESET}{BLUE}]{RESET}"

    label_str = ""
    if label:
        color = GREEN if label == "Yes" else RED
        label_str = f"  {BOLD}{color}=> {label}{RESET}"

    print(f"{prefix}{connector}{feature_str} = {value_str}  {entropy_str}  {S_str}{label_str}")

    for i, child in enumerate(children):
        is_last_child = (i == len(children) - 1)
        display_tree(
            node     = child,
            prefix   = prefix + (extension if not is_root else "    "),
            is_last  = is_last_child,
            is_root  = False
        )
        
display_tree(node=root)


┌── [outlook] = ROOT  H=0.94  S=[9Yes, 5No]
    ├── outlook = 'sunny'  H=0.97  S=[2Yes, 3No]
    │   └── temp = ROOT  H=0.97  S=[2Yes, 3No]
    │       ├── temp = 'hot'  H=0  S=[0Yes, 2No]  => No
    │       ├── temp = 'mild'  H=1  S=[1Yes, 1No]  => Yes
    │       └── temp = 'cool'  H=0  S=[1Yes, 0No]  => Yes
    ├── outlook = 'overcast'  H=0  S=[4Yes, 0No]  => Yes
    └── outlook = 'rain'  H=0.97  S=[3Yes, 2No]
        └── temp = ROOT  H=0.97  S=[3Yes, 2No]
            ├── temp = 'mild'  H=0.92  S=[2Yes, 1No]  => Yes
            └── temp = 'cool'  H=1  S=[1Yes, 1No]  => Yes
